In [1]:
import os
import sys
import site
import glob
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"

# 兼容 Kaggle 上两种可能的 utility script 路径
BASE_CANDIDATES = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script",
]

for base in BASE_CANDIDATES:
    if os.path.exists(base):
        site.addsitedir(base)
        sys.path.insert(0, base)
        print("Added utility path:", base)

# 自动寻找 CUTLASS python_packages 路径
cutlass_candidates = glob.glob(
    "/kaggle/usr/lib/notebooks/ryanholbrook/**/nvidia_cutlass_dsl/python_packages",
    recursive=True
)

for p in cutlass_candidates:
    if os.path.exists(p):
        site.addsitedir(p)
        sys.path.insert(0, p)
        print("Added CUTLASS path:", p)

# 修复 ptxas / ptxas-blackwell 权限
for name in ["ptxas", "ptxas-blackwell"]:
    found = glob.glob(
        f"/kaggle/usr/lib/notebooks/ryanholbrook/**/triton/backends/nvidia/bin/{name}",
        recursive=True
    )

    if found:
        src = found[0]
        dst = f"/tmp/{name}"
        shutil.copy(src, dst)
        os.chmod(dst, 0o755)

        if name == "ptxas":
            os.environ["TRITON_PTXAS_PATH"] = dst
        else:
            os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

        print(f"Prepared {name}:", dst)

# 让 /tmp 优先被搜索
os.environ["PATH"] = "/tmp:" + os.environ.get("PATH", "")

print("Environment initialized.")
print("TRITON_PTXAS_PATH =", os.environ.get("TRITON_PTXAS_PATH"))
print("TRITON_PTXAS_BLACKWELL_PATH =", os.environ.get("TRITON_PTXAS_BLACKWELL_PATH"))

Added utility path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script
Added CUTLASS path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages
Prepared ptxas: /tmp/ptxas
Prepared ptxas-blackwell: /tmp/ptxas-blackwell
Environment initialized.
TRITON_PTXAS_PATH = /tmp/ptxas
TRITON_PTXAS_BLACKWELL_PATH = /tmp/ptxas-blackwell


In [2]:
!/tmp/ptxas --version || true
!/tmp/ptxas-blackwell --version || true

ptxas: NVIDIA (R) Ptx optimizing assembler
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:21:21_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
ptxas-blackwell: NVIDIA (R) Ptx optimizing assembler
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_May_27_02:18:05_PDT_2025
Cuda compilation tools, release 12.9, V12.9.86
Build cuda_12.9.r12.9/compiler.36037853_0


In [3]:
import torch

print("CUDA device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

CUDA device count: 1
Device name: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True


In [4]:
import polars as pl

train = pl.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")

print(train.shape)
train.head()

(9500, 3)


id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [5]:
for i in range(3):
    row = train.row(i, named=True)
    print("=" * 100)
    print(row["prompt"][-800:])
    print("\nANSWER:", row["answer"])

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100

ANSWER: 10010111
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
10001110 -> 00100110
10011001 -> 01000100
01100100 -> 00010001
10000010 -> 00001010
00011011 -> 01001100
00111010 -> 10011100
01101111 -> 00110111
10010110 -> 01011010
00001010 -> 00101100

Now, determine the output for: 11100000

ANSWER: 01000011
In Al

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Using dtype:", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 关键：单卡加载，避免 device_map="auto" 把参数放到 cpu/disk/meta
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map={"": 0},
        trust_remote_code=True,
        dtype=DTYPE,
        low_cpu_mem_usage=True,
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map={"": 0},
        trust_remote_code=True,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
    )

print("Model loaded.")

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Using dtype: torch.bfloat16


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded.


In [7]:
LORA_RANK = 4

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=8,
    target_modules=r".*\.(in_proj|out_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 1,819,392 || all params: 31,579,756,736 || trainable%: 0.0058


In [8]:
from torch.utils.data import Dataset, DataLoader

model.config.use_cache = False
model.train()

# 开启 gradient checkpointing，省显存
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

MAX_TRAIN_SAMPLES = 500
MAX_LENGTH = 1024
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
LR = 2e-4

train_pd_small = train.to_pandas().sample(
    n=min(MAX_TRAIN_SAMPLES, train.shape[0]),
    random_state=42
).reset_index(drop=True)


class PromptCompletionDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=1024):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = str(row["prompt"]).rstrip() + "\n"
        answer = str(row["answer"]).strip() + self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=False,
        )["input_ids"]

        answer_ids = self.tokenizer(
            answer,
            add_special_tokens=False,
        )["input_ids"]

        # 保证 answer 不被截断；如果太长，优先保留 answer
        if len(answer_ids) >= self.max_length:
            input_ids = answer_ids[:self.max_length]
            labels = input_ids.copy()
        else:
            max_prompt_len = self.max_length - len(answer_ids)
            prompt_ids = prompt_ids[-max_prompt_len:]

            input_ids = prompt_ids + answer_ids
            labels = [-100] * len(prompt_ids) + answer_ids.copy()

        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    attention_mask = []
    labels = []

    for x in batch:
        pad_len = max_len - len(x["input_ids"])

        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


train_dataset = PromptCompletionDataset(
    train_pd_small,
    tokenizer,
    max_length=MAX_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
)

optimizer.zero_grad()

for step, batch in enumerate(train_loader, start=1):
    batch = {k: v.to("cuda:0") for k, v in batch.items()}

    with torch.amp.autocast("cuda", dtype=DTYPE):
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS

    loss.backward()

    if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
        optimizer.step()
        optimizer.zero_grad()

    print(f"step {step}/{len(train_loader)} | loss = {loss.item() * GRAD_ACCUM_STEPS:.4f}")

print("Training finished.")

step 1/500 | loss = 3.9453
step 2/500 | loss = 2.1631
step 3/500 | loss = 4.5217
step 4/500 | loss = 5.6974
step 5/500 | loss = 9.7953
step 6/500 | loss = 2.2784
step 7/500 | loss = 6.4863
step 8/500 | loss = 4.7402
step 9/500 | loss = 6.0463
step 10/500 | loss = 2.2259
step 11/500 | loss = 4.5590
step 12/500 | loss = 9.2872
step 13/500 | loss = 2.0858
step 14/500 | loss = 4.8482
step 15/500 | loss = 6.7016
step 16/500 | loss = 5.6972
step 17/500 | loss = 4.9746
step 18/500 | loss = 4.7762
step 19/500 | loss = 6.8090
step 20/500 | loss = 4.3591
step 21/500 | loss = 5.5298
step 22/500 | loss = 5.4996
step 23/500 | loss = 2.0910
step 24/500 | loss = 1.7964
step 25/500 | loss = 4.4457
step 26/500 | loss = 5.1396
step 27/500 | loss = 3.5690
step 28/500 | loss = 1.7752
step 29/500 | loss = 6.0894
step 30/500 | loss = 6.5933
step 31/500 | loss = 4.0143
step 32/500 | loss = 3.6671
step 33/500 | loss = 3.7566
step 34/500 | loss = 5.5246
step 35/500 | loss = 1.9079
step 36/500 | loss = 3.5311
s

In [9]:
import os
import shutil

ADAPTER_DIR = "/kaggle/working/adapter"

shutil.rmtree(ADAPTER_DIR, ignore_errors=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)

print("Saved adapter to:", ADAPTER_DIR)

!ls -lh /kaggle/working/adapter

Saved adapter to: /kaggle/working/adapter
total 7.0M
-rw-r--r-- 1 root root 1.1K May 16 09:34 adapter_config.json
-rw-r--r-- 1 root root 7.0M May 16 09:34 adapter_model.safetensors
-rw-r--r-- 1 root root 5.2K May 16 09:34 README.md


In [10]:
import subprocess
import os

zip_path = "/kaggle/working/submission.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

subprocess.run(
    "cd /kaggle/working/adapter && zip -r /kaggle/working/submission.zip .",
    shell=True,
    check=True,
)

print("Created:", zip_path)

!ls -lh /kaggle/working/submission.zip

  adding: adapter_model.safetensors (deflated 29%)
  adding: adapter_config.json (deflated 55%)
  adding: README.md (deflated 66%)
Created: /kaggle/working/submission.zip
-rw-r--r-- 1 root root 5.0M May 16 09:34 /kaggle/working/submission.zip


In [11]:
print('Done.')

Done.


In [12]:
!unzip -l /kaggle/working/submission.zip

Archive:  /kaggle/working/submission.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
  7289768  2026-05-16 09:34   adapter_model.safetensors
     1025  2026-05-16 09:34   adapter_config.json
     5312  2026-05-16 09:34   README.md
---------                     -------
  7296105                     3 files
